In [1]:
!pip install ipykernel numpy pandas matplotlib f5-tts ipywidgets ffmpeg torch torchaudio transformers

In [ ]:
# imports
from datasets import load_dataset
from importlib.resources import files
from f5_tts.api import F5TTS
from huggingface_hub import login
import torch
import pandas as pd
from pathlib import Path
import soundfile as sf
from tqdm.auto import tqdm

: 

In [18]:
# Use F5-TTS API to generate speech from text

tts = F5TTS()

wav, sr, spec = tts.infer(
    ref_file=str(files("data").joinpath("basic_ref_en.wav")),
    ref_text="some call me nature, others call me mother nature.",
    gen_text="I am a resident of the earth, and I am here to stay.",
    file_wave=str(files("data").joinpath("api_out.wav")),
    file_spec=str(files("data").joinpath("api_out.png")),
    seed=None,
)

Download Vocos from huggingface charactr/vocos-mel-24khz

vocab :  /home/ashish/work/notes/.venv/lib/python3.12/site-packages/f5_tts/infer/examples/vocab.txt
token :  custom
model :  /home/ashish/.cache/huggingface/hub/models--SWivid--F5-TTS/snapshots/84e5a410d9cead4de2f847e7c9369a6440bdfaca/F5TTS_v1_Base/model_1250000.safetensors 

Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   some call me nature, others call me mother nature. 
gen_text 0 I am a resident of the earth, and I am here to stay.


Generating audio in 1 batches...


100%|██████████| 1/1 [01:02<00:00, 62.88s/it]


In [5]:
wav, sr, spec = tts.infer(
    ref_file=str(files("data").joinpath("basic_ref_en.wav")),
    ref_text="some call me nature, others call me mother nature.",
    gen_text="मॆरा नाम आशिष है, और मैं पृथ्वी का निवासी हूँ।",
    file_wave=str(files("data").joinpath("api_out_hindi.wav")),
    file_spec=str(files("data").joinpath("api_out_hindi.png")),
    seed=None,
)

Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   some call me nature, others call me mother nature. 
gen_text 0 मॆरा नाम आशिष है, और मैं पृथ्वी का निवासी हूँ।


Generating audio in 1 batches...


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [02:02<00:00, 122.27s/it]


In [2]:
## Import indicVoices dataset for hindi language
dataset = load_dataset("ai4bharat/IndicVoices", "hindi", split="train")
# Keep only audio_filepath and text columns
dataset = dataset.select_columns(["audio_filepath", "text"])
print(dataset)

Resolving data files:   0%|          | 0/88 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/81 [00:00<?, ?it/s]

Dataset({
    features: ['audio_filepath', 'text'],
    num_rows: 383004
})


In [3]:
# Go through the text to identify all the graphemes in the dataset and save in a .txt file
graphemes = set()
for text in tqdm(dataset["text"]):
    for char in text:
        graphemes.add(char)

graphemes = sorted(list(graphemes))
print(graphemes)

  0%|          | 0/383004 [00:00<?, ?it/s]

[' ', '<', '>', 'b', 'e', 'g', 'i', 'l', 'n', 't', 'u', 'ँ', 'ं', 'ः', 'अ', 'आ', 'इ', 'ई', 'उ', 'ऊ', 'ऋ', 'ऍ', 'ए', 'ऐ', 'ऑ', 'ओ', 'औ', 'क', 'ख', 'ग', 'घ', 'ङ', 'च', 'छ', 'ज', 'झ', 'ञ', 'ट', 'ठ', 'ड', 'ढ', 'ण', 'त', 'थ', 'द', 'ध', 'न', 'प', 'फ', 'ब', 'भ', 'म', 'य', 'र', 'ऱ', 'ल', 'ळ', 'व', 'श', 'ष', 'स', 'ह', '़', 'ऽ', 'ा', 'ि', 'ी', 'ु', 'ू', 'ृ', 'ॅ', 'े', 'ै', 'ॉ', 'ो', 'ौ', '्', 'ॐ']


In [12]:
# Load a few examples to verify
for i in range(3):
    audio = dataset[i]["audio_filepath"]
    audio_array = audio["array"]
    sample_rate = audio["sampling_rate"]
    text = dataset[i]["text"]
    print(f"\nSample {i}:")
    print(f"  Text: {text}")
    print(f"  Audio shape: {audio_array.shape}")
    print(f"  Sample rate: {sample_rate}")
    print(f"  Duration: {len(audio_array)/sample_rate:.2f} seconds")


Sample 0:
  Text: घर
  Audio shape: (6464,)
  Sample rate: 16000
  Duration: 0.40 seconds

Sample 1:
  Text: तीन
  Audio shape: (46544,)
  Sample rate: 16000
  Duration: 2.91 seconds

Sample 2:
  Text: आज का समय बहुत ही डिजिटल समय हो गया है पहले क्या था कि पहले मोबाइल सबके हाथ में नहीं था और नया नया मोबाइल नहीं आया था छोटा मोबाइल था
  Audio shape: (143872,)
  Sample rate: 16000
  Duration: 8.99 seconds


In [ ]:
# Create output directory for training data
train_data_dir = Path("/home/ashish/work/notes/mtech/projects/data/f5_tts_hindi")
train_data_dir.mkdir(parents=True, exist_ok=True)

wavs_dir = train_data_dir / "wavs"
wavs_dir.mkdir(exist_ok=True)

# Process a subset of the dataset for finetuning
# Using first 500 samples for quick finetuning (adjust as needed)
max_samples = 500000

print(f"Processing {max_samples} samples for finetuning...")

records = []
for i in tqdm(range(max_samples), desc="Extracting audio"):
    audio = dataset[i]["audio_filepath"]
    audio_array = audio["array"]
    sample_rate = audio["sampling_rate"]
    text = dataset[i]["text"]
    
    # Save audio file
    wav_path = wavs_dir / f"hindi_{i:05d}.wav"
    sf.write(wav_path, audio_array, sample_rate)
    
    records.append({
        "audio_file": str(wav_path),
        "text": text
    })


Processing 500000 samples for finetuning...


Extracting audio:   0%|          | 0/500000 [00:00<?, ?it/s]

IndexError: Invalid key: 383004 is out of bounds for size 383004

In [ ]:
# Create CSV metadata
metadata_csv = train_data_dir / "metadata.csv"
df = pd.DataFrame(records)
df.to_csv(metadata_csv, sep="|", index=False, header=True)

print(f"\nCreated metadata.csv with {len(df)} records")
print(f"Sample records:")
print(df.head(3))


Created metadata.csv with 383004 records
Sample records:
                                          audio_file  \
0  /home/ashish/work/notes/mtech/projects/data/f5...   
1  /home/ashish/work/notes/mtech/projects/data/f5...   
2  /home/ashish/work/notes/mtech/projects/data/f5...   

                                                text  
0                                                 घर  
1                                                तीन  
2  आज का समय बहुत ही डिजिटल समय हो गया है पहले क्...  


In [ ]:
# Now prepare the dataset for F5-TTS training format
# This creates the processed training data
output_dir = train_data_dir / "processed"
output_dir.mkdir(exist_ok=True)

# run following command from terminal to prepare the dataset for F5-TTS training
# python3 f5_tts/train/datasets/prepare_csv_wavs.py ../../notes/mtech/projects/data/f5_tts_hindi/metadata.csv ../../notes/mtech/projects/data/f5_tts_hindi/processed/
# Additionally manually prepare a vocab.txt file to add all hindi characters to it


Preparing dataset for F5-TTS training...


TypeError: 'module' object is not callable

In [ ]:
## Run finetuning on the prepared dataset

# First, set up accelerate config (if not already done)
# accelerate config

# Then launch the finetuning with the base model
# Adjust the batch size and other hyperparameters as needed

# For finetuning, you can use the Gradio interface:
# from f5_tts.train.finetune_gradio import main as finetune_gradio
# finetune_gradio()

# Or use the command line (run in terminal):
# accelerate launch src/f5_tts/train/train.py --config-name F5TTS_v1_Base.yaml \
#     ++datasets.train_batch_size=16 \
#     ++model.ckpt_path=/path/to/pretrained/model.safetensors

# Example training command (run this in terminal):
print("""
To finetune F5-TTS on your Hindi dataset, run the following in terminal:

1. First configure accelerate:
   accelerate config

2. Then launch training:
   cd /home/ashish/work/notes
   source .venv/bin/activate
   accelerate launch src/f5_tts/train/train.py --config-name F5TTS_v1_Base.yaml \\
       ++datasets.train_batch_size=16 \\
       ++model.ckpt_path=/home/ashish/.cache/huggingface/hub/models--SWivid--F5-TTS/snapshots/84e5a410d9cead4de2f847e7c9369a6440bdfaca/F5TTS_v1_Base/model_1250000.safetensors

Or use the Gradio interface for easier finetuning:
   f5-tts_finetune-gradio
""")

In [2]:
fine_tuned_tts_model_ckpt = "model_5000.pt"
tts_hindi = F5TTS(ckpt_file=fine_tuned_tts_model_ckpt, vocab_file="/home/ashish/work/F5-TTS/data/hindi2_custom/vocab.txt")

Download Vocos from huggingface charactr/vocos-mel-24khz

vocab :  /home/ashish/work/F5-TTS/data/hindi2_custom/vocab.txt
token :  custom
model :  model_5000.pt 



In [3]:
wav, sr, spec = tts_hindi.infer(
    ref_file=str(files("data").joinpath("basic_ref_en.wav")),
    ref_text="some call me nature, others call me mother nature.",
    gen_text="मॆरा नाम आशिष है, और मैं पृथ्वी का निवासी हूँ।",
    file_wave=str(files("data").joinpath("api_out_hindi.wav")),
    file_spec=str(files("data").joinpath("api_out_hindi.png")),
    seed=None,
)

Converting audio...
Using custom reference text...

ref_text   some call me nature, others call me mother nature. 
gen_text 0 मॆरा नाम आशिष है, और मैं पृथ्वी का निवासी हूँ।


Generating audio in 1 batches...


100%|██████████| 1/1 [02:10<00:00, 130.19s/it]


In [2]:
torch.backends.mps.is_available()

False